In [ ]:
# @title
import pandas as pd
import json
import os
import glob

# Определяем имя выходного файла
output_xlsx_file = 'combined_xg_data.xlsx'

# Определяем желаемый порядок столбцов для итогового XLSX-файла
final_output_columns_order = [
    "match_id",
    "player_name",
    "minsPlayed",
    "epv_carry_n",
    "epv_carry_p",
    "epv_open_play_acc_n",
    "epv_open_play_acc_p",
    "epv_open_play_inacc_intercept",
    "epv_open_play_inacc_offside",
    "epv_open_play_inacc_out",
    "epv_set_piece_corner_acc_n",
    "epv_set_piece_corner_acc_p",
    "epv_set_piece_corner_inacc",
    "epv_set_piece_freekick_acc_n",
    "epv_set_piece_freekick_acc_p",
    "epv_set_piece_freekick_inacc",
    "epv_set_piece_throw_acc_n",
    "epv_set_piece_throw_acc_p",
    "epv_set_piece_throw_inacc"
]

# Создаем сопоставление имен столбцов JSON с желаемыми именами столбцов в выходном файле
# Предполагается, что JSON имеет суффисы _MINUS и _PLUS, где в выходном файле должны быть _n и _p
json_to_output_column_map = {
    "match_id": "match_id",
    "player_name": "player_name",
    "minsPlayed": "minsPlayed",
    "Carry_BETA.MINUS": "epv_carry_n",
    "Carry_BETA.PLUS": "epv_carry_p",
    "Open_Play.Accurate.MINUS": "epv_open_play_acc_n",
    "Open_Play.Accurate.PLUS": "epv_open_play_acc_p",
    "Open_Play.Inaccurate.Intercepted": "epv_open_play_inacc_intercept",
    "Open_Play.Inaccurate.To_Offside": "epv_open_play_inacc_offside",
    "Open_Play.Inaccurate.To_Out": "epv_open_play_inacc_out",
    "Set_Piece.Corner.Accurate_MINUS": "epv_set_piece_corner_acc_n",
    "Set_Piece.Corner.Accurate_PLUS": "epv_set_piece_corner_acc_p",
    "Set_Piece.Corner.Inaccurate": "epv_set_piece_corner_inacc",
    "Set_Piece.Free_Kick.Accurate_MINUS": "epv_set_piece_freekick_acc_n",
    "Set_Piece.Free_Kick.Accurate_PLUS": "epv_set_piece_freekick_acc_p",
    "Set_Piece.Free_Kick.Inaccurate": "epv_set_piece_freekick_inacc",
    "Set_Piece.Throw_In.Accurate_MINUS": "epv_set_piece_throw_acc_n",
    "Set_Piece.Throw_In.Accurate_PLUS": "epv_set_piece_throw_acc_p",
    "Set_Piece.Throw_In.Inaccurate": "epv_set_piece_throw_inacc"
}

all_processed_data = []

# Находим все JSON-файлы, начинающиеся с 'xg_' в текущей директории
json_files = glob.glob('xg_*.json')

if not json_files:
    print("Ошибка: JSON-файлы, начинающиеся с 'xg_', не найдены в текущей директории.")
else:
    for input_json_file in json_files:
        try:
            # Извлекаем match_id из имени файла
            try:
                match_id_part = os.path.basename(input_json_file).split('_', 1)[1]
                match_id = match_id_part.split('.')[0]
            except IndexError:
                match_id = 'unknown' # Устанавливаем значение по умолчанию, если формат файла не соответствует

            # Читаем содержимое JSON-файла
            with open(input_json_file, 'r') as f:
                json_data = json.load(f)

            # Убеждаемся, что ключ 'records' существует и является списком
            if not isinstance(json_data, dict) or 'records' not in json_data or not isinstance(json_data['records'], list):
                print(f"Предупреждение: Неподдерживаемая структура JSON в файле '{input_json_file}'. Пропускаем файл.")
                continue

            records = json_data['records']

            for record in records:
                player_data = {
                    "match_id": match_id,
                    "player_name": record.get("playerName"),
                    "minsPlayed": record.get("minsPlayed") # Добавляем minsPlayed
                }

                # Извлекаем и "выравниваем" данные 'classes'
                for class_item in record.get("classes", []):
                    class_name = class_item.get("className")
                    value = class_item.get("value")
                    if class_name:
                        player_data[class_name] = value

                all_processed_data.append(player_data)

        except FileNotFoundError:
            print(f"Ошибка: Файл '{input_json_file}' не найден. Убедитесь, что он загружен.")
        except Exception as e:
            print(f"Произошла ошибка при обработке файла '{input_json_file}': {e}")

    if all_processed_data:
        df = pd.DataFrame(all_processed_data)

        # Определяем, какие столбцы JSON (после "выравнивания") из нашей карты присутствуют в DataFrame
        # и создаем для них словарь переименования
        present_json_cols_to_rename = {}
        for json_col_key, output_col_name in json_to_output_column_map.items():
            if json_col_key in df.columns:
                present_json_cols_to_rename[json_col_key] = output_col_name

        # Переименовываем столбцы в DataFrame
        df.rename(columns=present_json_cols_to_rename, inplace=True)

        # Выбираем и переупорядочиваем столбцы в соответствии с 'final_output_columns_order'
        # Включаем только те столбцы, которые фактически присутствуют в (потенциально переименованном) DataFrame
        final_cols_for_selection = [col for col in final_output_columns_order if col in df.columns]
        df_filtered = df[final_cols_for_selection]

        # Проверяем, не были ли пропущены какие-либо исходные желаемые столбцы даже после сопоставления
        missing_output_cols = set(final_output_columns_order) - set(df_filtered.columns)
        if missing_output_cols:
            print(f"Предупреждение: Некоторые запрошенные выходные столбцы не были найдены в JSON-файлах (даже после сопоставления и обработки). Отсутствующие столбцы: {missing_output_cols}")

        # Сохраняем отфильтрованный и переупорядоченный DataFrame в Excel-файл
        df_filtered.to_excel(output_xlsx_file, index=False)

        print(f"Успешно преобразованы {len(json_files)} файлов JSON в '{output_xlsx_file}' с указанными столбцами и порядком.")
    else:
        print("Не удалось обработать данные ни из одного JSON-файла.")

Успешно преобразованы 10 файлов JSON в 'combined_xg_data.xlsx' с указанными столбцами и порядком.


In [ ]:
# @title
import pandas as pd
import json
import os
import re
from google.colab import drive
drive.mount('/content/drive')

# Меняйте только эту маску: 'ps_epl_*.json' или 'ps_rpl_*.json'
json_file_mask = 'ps_rpl_*.json'

league_search = re.fullmatch(r'ps_([er]pl)_\*\.json', json_file_mask)
if not league_search:
    raise ValueError("Используйте маску 'ps_epl_*.json' или 'ps_rpl_*.json'.")

league = league_search.group(1)
excel_file_name = f'PlayerStats{league.upper()}27.xlsx'
json_file_pattern = re.compile(
    re.escape(json_file_mask).replace(r'\*', r'(\d+)')
)

# Загружаем справочник игроков.
# Google Диск должен быть предварительно подключен в Google Colab.
players_file_path = '/content/drive/MyDrive/Players.xlsx'
players_sheet_name = 'e' if league == 'epl' else 'r'

if not os.path.isfile(players_file_path):
    raise FileNotFoundError(
        f"Файл справочника '{players_file_path}' не найден. "
        "Проверьте путь и подключение Google Диска."
    )

players_reference_df = pd.read_excel(
    players_file_path,
    sheet_name=players_sheet_name,
    usecols=[0, 1],
    dtype=object,
    engine='openpyxl'
)

if players_reference_df.shape[1] != 2:
    raise ValueError(
        f"На листе '{players_sheet_name}' файла Players.xlsx "
        "должны быть два столбца: имя игрока и player_id."
    )

players_reference_df.columns = ['player_name', 'player_id']

# При повторяющемся имени используем первое совпадение, как ВПР.
players_reference_df = (
    players_reference_df
    .dropna(subset=['player_name'])
    .drop_duplicates(subset=['player_name'], keep='first')
)

player_id_mapping = players_reference_df.set_index('player_name')['player_id']

# Соответствие между ключами JSON и названиями столбцов XLSX
column_mapping = {
    'name': 'plsyer_name',
    'minutesPlayed': 'minutes_played',
    'goals': 'goals',
    'goalAssist': 'assists',
    'totalClearance': 'clearances',
    'ballRecovery': 'recovery',
    'outfielderBlock': 'blocked_shots',
    'interceptionWon': 'interceptions',
    'totalTackle': 'total_tackles',
    'wonTackle': 'tackles_won',
    'challengeLost': 'dribbled_past',
    'duelLost': 'ground_duels',
    'duelWon': 'ground_duels_won',
    'aerialLost': 'aerial_duels',
    'aerialWon': 'aerial_duels_won',
    'possessionLostCtrl': 'possession_lost',
    'fouls': 'fouls',
    'wasFouled': 'was_fouled',
    'touches': 'touches',
    'unsuccessfulTouch': 'touches_unsuc',
    'totalPass': 'passes_total',
    'accuratePass': 'passes_accurate',
    'totalOppositionHalfPasses': 'passes_in_opp_total',
    'accurateOppositionHalfPasses': 'passes_in_opp_accurate',
    'totalOwnHalfPasses': 'passes_in_own_total',
    'accurateOwnHalfPasses': 'passes_in_own_accurate',
    'keyPass': 'key_passes',
    'totalCross': 'crosses_total',
    'accurateCross': 'crosses_accurate',
    'totalLongBalls': 'long_balls_total',
    'accurateLongBalls': 'long_balls_accurate',
    'onTargetScoringAttempt': 'shots_on_target',
    'shotOffTarget': 'shots_off_target',
    'blockedScoringAttempt': 'shots_blocked',
    'totalContest': 'dribble_att',
    'wonContest': 'dribble_succ',
    'rating': 'rating',
    'bigChanceCreated': 'big_chances_created',
    'bigChanceMissed': 'big_chances_missed',
    'totalOffside': 'offsides',
    'hitWoodwork': 'hit_woodwork',
    'clearanceOffLine': 'clearance_off_line',
    'ownGoals': 'own_goals',
    'errorLeadToAShot': 'errors_lead_to_shot',
    'errorLeadToAGoal': 'errors_lead_to_goal',
    'penaltyWon': 'penalty_won',
    'penaltyConceded': 'penalty_committed',
    'penaltyMiss': 'penalty_unrealized',
    'kilometersCovered': 'distance_covered',
    'topSpeed': 'top_speed',
    'numberOfSprints': 'number_of_sprints',
    'metersCoveredRunningKm': 'running',
    'metersCoveredHighSpeedRunningKm': 'high_speed_running',
    'metersCoveredSprintingKm': 'sprinting',
    'totalBallCarriesDistance': 'total_carrying_distance',
    'ballCarriesCount': 'carries',
    'progressiveBallCarriesCount': 'progressive_carries',
    'totalProgression': 'total_progression',
    'totalProgressiveBallCarriesDistance': 'progressive_carrying_distance',
    'bestBallCarryProgression': 'longest_progressive_carry',
    'substitute': 'is_substitute',
    'teamId': 'team_id'
}

# Столбцы итогового XLSX в нужном порядке
final_xlsx_columns_order = [
    'match_id', 'plsyer_name', 'player_id', 'team_id', 'minutes_played', 'goals', 'assists',
    'yellow_card', 'second_yellow_card', 'red_card', 'xg', 'xa', 'xGChain', 'xGBuildup',
    'clearances', 'recovery', 'blocked_shots', 'interceptions', 'total_tackles', 'tackles_won', 'dribbled_past',
    'ground_duels', 'ground_duels_won', 'aerial_duels', 'aerial_duels_won',
    'possession_lost', 'fouls', 'was_fouled', 'touches', 'touches_unsuc', 'passes_total',
    'passes_accurate', 'passes_in_opp_total', 'passes_in_opp_accurate', 'passes_in_own_total', 'passes_in_own_accurate', 'key_passes', 'crosses_total', 'crosses_accurate',
    'long_balls_total', 'long_balls_accurate', 'shots_on_target',
    'shots_off_target', 'shots_blocked', 'dribble_att', 'dribble_succ',
    'starting_player', 'position_on_field', 'rating', 'big_chances_created',
    'big_chances_missed', 'offsides', 'hit_woodwork', 'clearance_off_line',
    'own_goals', 'errors_lead_to_shot', 'errors_lead_to_goal', 'penalty_won',
    'penalty_committed', 'penalty_realized', 'penalty_unrealized', 'distance_covered', 'top_speed', 'number_of_sprints', 'running', 'high_speed_running', 'sprinting', 'total_carrying_distance', 'carries', 'progressive_carries', 'total_progression', 'progressive_carrying_distance', 'longest_progressive_carry'
]

# Замена идентификаторов команд
team_id_mapping = {
    42: 2,
    34: 18,
    40: 9,
    50: 13,
    38: 3,
    60: 11,
    32: 22,
    43: 15,
    41: 19,
    30: 8,
    35: 6,
    33: 5,
    7: 10,
    14: 12,
    96: 23,
    48: 17,
    44: 4,
    17: 1,
    11: 21,
    39: 7,
    2326: 7,
    285689: 9,
    24118: 16,
    362016: 13,
    2317: 15,
    2320: 6,
    2322: 10,
    2323: 4,
    34425: 3,
    2321: 2,
    5131: 14,
    7517: 17,
    2325: 1,
    322699: 20,
    2333: 8,
    2315: 5
}

# Столбцы, которые всегда должны оставаться пустыми
always_empty_cols = [
    'minutes_played', 'xg', 'xa', 'xGChain', 'xGBuildup'
]

# Столбцы для округления и отображения с двумя знаками после запятой
round_columns = [
    'distance_covered',
    'top_speed',
    'running',
    'high_speed_running',
    'sprinting',
    'total_carrying_distance',
    'carries',
    'progressive_carries',
    'total_progression',
    'progressive_carrying_distance',
    'longest_progressive_carry'
]

all_matches_data = []

# Вспомогательная функция для разворачивания данных игрока
def flatten_player_data(player_list):
    flat_data = []
    for player_dict in player_list:
        flattened_player = player_dict.copy()
        if 'player' in flattened_player:
            player_details = flattened_player.pop('player')
            flattened_player.update(player_details)
        flat_data.append(flattened_player)
    return flat_data

# Получаем список JSON-файлов выбранной лиги
json_files_to_process = [
    f for f in os.listdir('.')
    if json_file_pattern.fullmatch(f) and os.path.isfile(f)
]

if not json_files_to_process:
    print(f"Файлы '{json_file_mask}' не найдены в текущей директории. Проверьте имена файлов.")
else:
    for json_file in json_files_to_process:
        print(f"Обработка файла '{json_file}'...")

        match_id_search = json_file_pattern.fullmatch(json_file)
        if match_id_search:
            match_id = int(match_id_search.group(1))
        else:
            print(f"Не удалось извлечь match_id из имени файла '{json_file}'. Пропускаем.")
            continue

        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                df_raw = pd.read_json(f)

            home_players_data = df_raw.loc['players', 'home']
            away_players_data = df_raw.loc['players', 'away']

            # Разворачиваем данные игроков
            home_flat_players = flatten_player_data(home_players_data)
            away_flat_players = flatten_player_data(away_players_data)

            df_home_players = pd.DataFrame(home_flat_players)
            df_away_players = pd.DataFrame(away_flat_players)

            # Временный столбец для сортировки Home/Away
            df_home_players['team_category'] = 'Home'
            df_away_players['team_category'] = 'Away'

            match_df = pd.concat([df_home_players, df_away_players], ignore_index=True)

            match_df['match_id'] = match_id

            # Разворачиваем статистику
            if 'statistics' in match_df.columns and not match_df['statistics'].empty:
                stats_df = pd.json_normalize(
                    match_df['statistics'].apply(
                        lambda x: x if isinstance(x, dict) else {}
                    )
                )
                match_df = match_df.drop(columns=['statistics'])
                match_df = pd.concat([match_df, stats_df], axis=1)

            # Переименовываем столбцы
            match_df = match_df.rename(columns=column_mapping)

            # False = стартовый игрок, True = запасной.
            # При отсутствующем признаке сохраняем значение по умолчанию 0.
            if 'is_substitute' in match_df.columns:
                match_df['starting_player'] = match_df['is_substitute'].apply(
                    lambda x: 0 if pd.isna(x) else (0 if x else 1)
                )
                match_df = match_df.drop(columns=['is_substitute'])
            else:
                match_df['starting_player'] = 0

            # Добавляем отсутствующие итоговые столбцы
            for col in final_xlsx_columns_order:
                if col not in match_df.columns:
                    match_df[col] = pd.NA

            # Подставляем player_id по точному совпадению имени со справочником.
            # При отсутствии совпадения ячейка останется пустой.
            match_df['player_id'] = match_df['plsyer_name'].map(player_id_mapping)

            # Преобразуем итоговые числовые столбцы.
            # minutes_played пока сохраняем для фильтрации.
            # rating не заполняем нулями при отсутствии значения.
            # Имена и идентификаторы из справочника сохраняем без преобразования.
            for col in final_xlsx_columns_order:
                if col == 'plsyer_name':
                    match_df[col] = match_df[col].fillna('0')
                    continue

                if col in ['player_id', 'xg', 'xa', 'xGChain', 'xGBuildup', 'position_on_field']:
                    continue

                match_df[col] = pd.to_numeric(match_df[col], errors='coerce')

                if col != 'rating':
                    match_df[col] = match_df[col].fillna(0)

            # duelLost + duelWon
            match_df['ground_duels'] = (
                match_df['ground_duels'] + match_df['ground_duels_won']
            )

            # aerialLost + aerialWon
            match_df['aerial_duels'] = (
                match_df['aerial_duels'] + match_df['aerial_duels_won']
            )

            # Заменяем team_id однократно.
            # Значения, отсутствующие в словаре, остаются без изменений.
            match_df['team_id'] = match_df['team_id'].replace(team_id_mapping)

            # Удаляем игроков, не сыгравших ни одной минуты,
            # прежде чем очистить minutes_played
            match_df = match_df[match_df['minutes_played'] > 0].copy()

            # Принудительно очищаем указанные столбцы
            for col in always_empty_cols:
                match_df[col] = pd.NA

            # Позиция пустая для стартовых игроков, Sub — для запасных
            match_df['position_on_field'] = pd.Series(
                pd.NA, index=match_df.index, dtype='object'
            )
            match_df.loc[
                match_df['starting_player'] == 0,
                'position_on_field'
            ] = 'Sub'

            # Округляем значения до двух знаков после запятой
            match_df[round_columns] = match_df[round_columns].round(2)

            all_matches_data.append(match_df)

        except FileNotFoundError:
            print(f"Ошибка: Файл '{json_file}' не найден. Пропускаем.")
        except KeyError as e:
            print(f"Ошибка KeyError при обработке файла '{json_file}': {e}. Проверьте структуру JSON.")
        except Exception as e:
            print(f"Произошла непредвиденная ошибка при обработке файла '{json_file}': {e}")

# Объединяем данные всех матчей
if all_matches_data:
    final_output_df = pd.concat(all_matches_data, ignore_index=True)

    final_output_df['match_id'] = pd.to_numeric(
        final_output_df['match_id'], errors='coerce'
    )

    # Сначала Home, затем Away
    team_category_order = ['Home', 'Away']
    final_output_df['team_category'] = pd.Categorical(
        final_output_df['team_category'],
        categories=team_category_order,
        ordered=True
    )

    final_output_df = final_output_df.sort_values(
        by=['match_id', 'team_category'],
        ascending=[True, True],
        kind='stable'
    ).reset_index(drop=True)

    final_output_df = final_output_df.drop(columns=['team_category'])
    final_output_df = final_output_df[final_xlsx_columns_order]

    # Сохраняем XLSX и задаем формат 0.00 для нужных столбцов.
    # Пустые значения сохраняются как пустые ячейки.
    try:
        with pd.ExcelWriter(excel_file_name, engine='openpyxl') as writer:
            final_output_df.to_excel(
                writer,
                index=False,
                sheet_name='Sheet1',
                na_rep=''
            )

            worksheet = writer.sheets['Sheet1']

            if not final_output_df.empty:
                for col in round_columns:
                    column_index = final_output_df.columns.get_loc(col) + 1

                    for cells in worksheet.iter_cols(
                        min_col=column_index,
                        max_col=column_index,
                        min_row=2,
                        max_row=len(final_output_df) + 1
                    ):
                        for cell in cells:
                            cell.number_format = '0.00'

        print(f"Данные успешно обработаны и сохранены в '{excel_file_name}'")
    except Exception as e:
        print(f"Произошла ошибка при сохранении файла Excel: {e}")
else:
    print("Данные не были обработаны ни из одного JSON-файла.")

Mounted at /content/drive
Обработка файла 'ps_rpl_584.json'...
Данные успешно обработаны и сохранены в 'PlayerStatsRPL27.xlsx'


In [ ]:
import json
import re
import sqlite3
from pathlib import Path
import pandas as pd
from google.colab import drive

# Выберите маску файлов
JSON_MASK = 'ms_rpl_*.json'  # или 'ms_epl_*.json'
match = re.fullmatch(r'ms_(rpl|epl)_\*\.json', JSON_MASK)
if not match:
    raise ValueError("JSON_MASK должен быть 'ms_rpl_*.json' или 'ms_epl_*.json'.")
league = match.group(1)
OUTPUT = f'MatchStats{league.upper()}27.xlsx'
FILE_RE = re.compile(re.escape(JSON_MASK).replace(r'\*', r'(\d+)'))

# Подключаем Google Drive и выбираем базу лиги
drive.mount('/content/drive')
DB_NAME = 'rpl.db' if league == 'rpl' else 'FA_Premier_League.db'
DB_PATH = Path('/content/drive/MyDrive/Data_Bases') / DB_NAME
if not DB_PATH.is_file():
    raise FileNotFoundError(f'Не найдена база данных: {DB_PATH}')

# JSON: составные показатели "успешно/всего"
SPLIT_STATS = {
    'Long balls': ('Long balls', 'Accurate long balls'),
    'Crosses': ('Crosses', 'Accurate crosses'),
    'Ground duels': ('Ground duels', 'Ground duels won'),
    'Aerial duels': ('Aerial duels', 'Aerial duels won'),
    'Dribbles': ('Dribbles', 'Dribbles successful'),
    'Final third phase': ('Final third phase', 'Successful final third phase'),
}

# Итоговый столбец Excel -> точное имя показателя в JSON
STAT_MAPPING = {
    'corners': 'Corner kicks',
    'yellow_cards': 'Yellow cards',
    'red_cards': 'Red cards',
    'offsides': 'Offsides',
    'fouls': 'Fouls',
    'saves': 'Goalkeeper saves',
    'possession': 'Ball possession',
    'shots': 'Total shots',
    'shots_on_target': 'Shots on target',
    'hit_woodwork': 'Hit woodwork',
    'shots_off_target': 'Shots off target',
    'blocked_shots': 'Blocked shots',
    'shots_inside_box': 'Shots inside box',
    'shots_outside_box': 'Shots outside box',
    'passes': 'Passes',
    'accurate_passes': 'Accurate passes',
    'final_third_phase': 'Final third phase',
    'accurate_final_third_phase': 'Successful final third phase',
    'long_balls': 'Long balls',
    'accurate_long_balls': 'Accurate long balls',
    'crosses': 'Crosses',
    'accurate_crosses': 'Accurate crosses',
    'ground_duels': 'Ground duels',
    'ground_duels_won': 'Ground duels won',
    'aerial_duels': 'Aerial duels',
    'aerial_duels_won': 'Aerial duels won',
    'dribbles': 'Dribbles',
    'dribbles_succ': 'Dribbles successful',
    'throw_ins': 'Throw-ins',
    'through_balls': 'Through balls',
    'distance_covered': 'Distance covered',
    'number_of_sprints': 'Number of sprints',
    'touches_in_opp_box': 'Touches in penalty area',
    'final_third_entries': 'Final third entries',  # отдельный показатель, не Final third phase
    'fouled_in_final_third': 'Fouled in final third',
}

COLUMNS = [
    'match_id', 'team_id', 'goals', 'xg', 'ppda', 'deep', 'corners',
    'yellow_cards', 'red_cards', 'offsides', 'fouls', 'saves', 'possession',
    'shots', 'shots_on_target', 'hit_woodwork', 'shots_off_target',
    'blocked_shots', 'shots_inside_box', 'shots_outside_box', 'passes',
    'accurate_passes', 'final_third_phase', 'accurate_final_third_phase',
    'long_balls', 'accurate_long_balls', 'crosses', 'accurate_crosses',
    'ground_duels', 'ground_duels_won', 'aerial_duels', 'aerial_duels_won',
    'dribbles', 'dribbles_succ', 'throw_ins', 'through_balls',
    'distance_covered', 'number_of_sprints', 'touches_in_opp_box',
    'final_third_entries', 'fouled_in_final_third',
]


def parse_value(raw, numeric):
    """Как в предыдущем скрипте: сначала homeValue/awayValue, затем raw."""
    if numeric is not None:
        return numeric
    if raw is None:
        return None
    try:
        return int(raw)
    except (TypeError, ValueError):
        try:
            return float(raw)
        except (TypeError, ValueError):
            return raw


def parse_split(raw, numeric):
    """Из 'успешно/всего' получает (всего, успешно), как исходный скрипт."""
    successful = total = None
    if isinstance(raw, str):
        found = re.search(r'(\d+)\s*/\s*(\d+)', raw)
        if found:
            successful, total = map(int, found.groups())
        else:
            found = re.search(r'\d+', raw)
            if found:
                total = int(found.group())
    if total is None and numeric is not None:
        total = numeric
    if successful is None and total is not None:
        successful = 0
    return total, successful


def get_team_ids(db, match_id):
    row = db.execute(
        'SELECT team1_id, team2_id FROM Matches WHERE match_id = ?',
        (match_id,),
    ).fetchone()
    if row is None:
        raise ValueError(f'match_id={match_id} отсутствует в Matches')
    return row  # team1_id = Home, team2_id = Away


def read_match(json_path, match_id, team_ids):
    with json_path.open(encoding='utf-8') as file:
        data = json.load(file)

    aggregated = {}
    for period in data.get('statistics', []):
        if period.get('period') != 'ALL':
            continue
        for group in period.get('groups', []):
            for item in group.get('statisticsItems', []):
                name = item.get('name')
                if not name:
                    continue
                for side in ('home', 'away'):
                    raw, numeric = item.get(side), item.get(side + 'Value')
                    if name in SPLIT_STATS:
                        total_name, successful_name = SPLIT_STATS[name]
                        total, successful = parse_split(raw, numeric)
                        aggregated[f'{side}_{total_name}'] = total
                        aggregated[f'{side}_{successful_name}'] = successful
                    else:
                        value = parse_value(raw, numeric)
                        if value is not None:
                            aggregated[f'{side}_{name}'] = value
        break  # обрабатываем только период ALL

    if not aggregated:
        print(f'{json_path.name}: нет статистики периода ALL — пропуск')
        return []

    rows = []
    for side, team_id in zip(('home', 'away'), team_ids):
        row = {column: None for column in COLUMNS}
        row.update({'match_id': match_id, 'team_id': team_id})
        for excel_column, json_name in STAT_MAPPING.items():
            row[excel_column] = aggregated.get(f'{side}_{json_name}')
        rows.append(row)
    return rows


json_files = sorted(
    (p for p in Path('.').glob(JSON_MASK) if p.is_file()),
    key=lambda p: int(FILE_RE.fullmatch(p.name).group(1)),
)
if not json_files:
    raise SystemExit(f"Файлы '{JSON_MASK}' не найдены в текущей папке.")

all_rows = []
with sqlite3.connect(str(DB_PATH)) as db:
    for json_path in json_files:
        match_id = int(FILE_RE.fullmatch(json_path.name).group(1))
        try:
            team_ids = get_team_ids(db, match_id)
            rows = read_match(json_path, match_id, team_ids)
            all_rows.extend(rows)
            if rows:
                print(f'{json_path.name}: Home={team_ids[0]}, Away={team_ids[1]}')
        except Exception as error:
            print(f'Ошибка в {json_path.name}: {error}')

if all_rows:
    result = pd.DataFrame(all_rows).reindex(columns=COLUMNS)
    # Порядок в каждом матче: Home, затем Away (не зависит от team_id)
    result['_side_order'] = result.groupby('match_id').cumcount()
    result = result.sort_values(['match_id', '_side_order'], kind='stable')
    result = result.drop(columns='_side_order').reset_index(drop=True)
    result.to_excel(OUTPUT, index=False, engine='openpyxl')
    print(f'Готово: обработано {len(all_rows) // 2} матчей, файл {OUTPUT}')
else:
    print('Нет данных для сохранения.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ms_rpl_577.json: Home=8, Away=15
ms_rpl_578.json: Home=5, Away=14
ms_rpl_579.json: Home=20, Away=3
ms_rpl_580.json: Home=16, Away=1
ms_rpl_581.json: Home=13, Away=7
ms_rpl_582.json: Home=17, Away=9
ms_rpl_583.json: Home=6, Away=4
ms_rpl_584.json: Home=10, Away=2
Готово: обработано 8 матчей, файл MatchStatsRPL27.xlsx
